# 2026-11-24 Baselines: naive and seasonal naive on FOODS
First modelling notebook. The two simplest forecasts, so every later model has a bar to beat.

## Today's steps (standup tomorrow)
- [x] **Decide** the level (total, state or store) and write why this one, why not the others
- [x] **Do** load the daily series and hold out the last 28 days as the test window
- [x] **Chart** all 30 series, whole period, simple to read
- [x] **Do** naive: the forecast is yesterday's value
- [x] **Study** pick the error metrics and why
- [x] **Do** score naive
- [x] **Do** seasonal naive: 7 days ago
- [x] **Do** seasonal naive: 30 days ago
- [x] **Chart** both forecasts against the actual test days
- [x] **Do** score both and compare
- [x] **Observe** what does the gap between the two say about the weekly rhythm? (3 lines)
- [x] **Write** standup update: level, metric, both scores, one chart

**Carried in from EDA:** strong weekly cycle (lag 7 = 0.72, weekday explains about 51% of daily variation); monthly SNAP wave; no real yearly season; Wisconsin stores 1 and 2 have a level jump; only 2 of about 890 items are strongly seasonal.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import plotly.express as px, plotly.graph_objects as go
from m5food import data, plots
plots.set_style()

## Load

In [ ]:
sales = data.add_readable_names(data.load_sales_foods())
cal = data.load_calendar()
days = data.day_columns(sales)
dates = cal.set_index('d').loc[days, 'date'].values      # d_1 ... d_1941 -> dates
sales.shape

## Daily series at each level
Units per day: all FOODS, per state, per store, and per store × department (the forecasting level). Nothing is cleaned yet.

In [ ]:
idx = pd.DatetimeIndex(dates, name='date')
daily_total = pd.Series(sales[days].sum().to_numpy(), index=idx, name='All FOODS')
daily_state = sales.groupby('state')[days].sum().T.set_axis(idx)
daily_store = sales.groupby('store')[days].sum().T.set_axis(idx)
daily_store_dept = sales.groupby(['store', 'dept'])[days].sum().T.set_axis(idx)     # 30 columns: (store, dept)
print('total:', daily_total.shape, '| states:', daily_state.shape, '| stores:', daily_store.shape,
      '| store x dept:', daily_store_dept.shape)
print('last day in data:', idx[-1].date())

## Step 1: decide the level
*(after reading Hyndman ch. 11)*

- Started with: store? dept? item? total? Thought I had to pick one.
- Book says no. Pick a level, then add up / split down. Reconcile later (11.3).
- My data isn't a clean tree. Store and dept cross each other, so it's grouped.
- Item level = finest, but ~14,000 series, mostly zeros. Too noisy to start there.
- Total = smooth, but hides store and dept differences (weekly rhythm, Wisconsin jumps).
- So the middle: **store × dept, 30 series.** Small enough to check by eye, big enough to keep the patterns.
- Then: add up → store, total. Split down → items, later.

**Level:** store × dept (30), middle-out.
**Ask Chiranjit:** at what level does the client actually decide?

In [ ]:
LEVEL = 'store_dept'     # decided in Step 1: 30 daily series (10 stores x 3 food departments), middle-out
series = {'total': daily_total.to_frame(), 'state': daily_state, 'store': daily_store,
          'store_dept': daily_store_dept}[LEVEL]
series.tail()


## Step 2: hold out the last 28 days

In [ ]:
HORIZON = 28
train, test = series.iloc[:-HORIZON], series.iloc[-HORIZON:]
print(f'train: {train.index[0].date()} to {train.index[-1].date()} ({len(train)} days)')
print(f'test:  {test.index[0].date()} to {test.index[-1].date()} ({len(test)} days)')

## Step 3: look at the 30 series first

In [ ]:
# All 30 series, whole period. Rows = stores, columns = food departments.
# Each point = average units per day in that week (daily is too spiky to read at this size).
# Same y-scale within a column, so stores can be compared inside a department.
from plotly.subplots import make_subplots
STATE_COLOR = {'California': plots.PALETTE[0], 'Texas': plots.PALETTE[1], 'Wisconsin': plots.PALETTE[2]}
stores = sorted(daily_store_dept.columns.get_level_values(0).unique())
depts = sorted(daily_store_dept.columns.get_level_values(1).unique())
weekly = daily_store_dept.resample('W').mean()

fig = make_subplots(rows=len(stores), cols=len(depts), shared_xaxes=True, shared_yaxes='columns',
                    column_titles=[f'<b>{d}</b>' for d in depts], row_titles=stores,
                    vertical_spacing=0.012, horizontal_spacing=0.04)
for r, store in enumerate(stores, start=1):
    for c, dept in enumerate(depts, start=1):
        y = weekly[(store, dept)]
        fig.add_trace(go.Scatter(x=y.index, y=y, mode='lines', showlegend=False,
                                 line=dict(color=STATE_COLOR[store.split(' store ')[0]], width=1.4),
                                 hovertemplate=f'{store}, {dept}<br>week of %{{x|%d %b %Y}}: '
                                               '%{y:,.0f} a day<extra></extra>'), row=r, col=c)
fig.update_yaxes(rangemode='tozero', nticks=3, tickfont=dict(size=9))
fig.update_annotations(selector=dict(textangle=90), textangle=0, xanchor='left', font=dict(size=10))
fig.update_layout(title='<b>The 30 series we will forecast: 10 stores × 3 food departments</b><br>'
                        '<span style="font-size:13px;color:#444">Average units sold per day, week by week, 2011 to 2016. '
                        'Colour = state (blue California, orange Texas, green Wisconsin). Same scale down each column.</span>',
                  template='simple_white', height=1500, margin=dict(t=110, r=150))
fig.show()

**What I see (draft)**
- Food dept 3 is the big one. ~1,000 to 2,900 a day per store. Dept 1 and 2 are a few hundred.
- Most lines just drift up slowly. Fine for naive.
- But the store jumps sit in **Food dept 2**: Wisconsin store 1 grew ~11x, Wisconsin store 2 ~5x, California store 2 ~3x.
- Food dept 1 grew almost everywhere (+40% to +150%).
- So the 30 series aren't alike. Good reason to forecast them one by one, not as a total.

## Step 3b: are the 30 series stationary? KPSS and ADF, one at a time

In [ ]:
# Each test on its own, on the 30 RAW training series (no differencing yet).
# KPSS: starts from "stationary".     p below 0.05 -> red (not stationary), else green. (Hyndman's book uses this one.)
# ADF:  starts from "NOT stationary". p below 0.05 -> green (stationary),  else red.
# KPSS only reports p between 0.01 and 0.10, so 0.01 means "0.01 or less" and 0.10 means "0.10 or more".
import warnings
from statsmodels.tsa.stattools import adfuller, kpss

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    kpss_p = pd.Series({c: kpss(train[c], regression='c', nlags='auto')[1] for c in train})
    adf_p = pd.Series({c: adfuller(train[c], autolag='AIC')[1] for c in train})

tests = pd.DataFrame({'KPSS p': kpss_p, 'KPSS says': np.where(kpss_p < 0.05, 'Not stationary', 'Stationary'),
                      'ADF p': adf_p, 'ADF says': np.where(adf_p < 0.05, 'Stationary', 'Not stationary')})
print(tests[['KPSS says', 'ADF says']].apply(pd.Series.value_counts).fillna(0).astype(int))

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=['<b>KPSS</b> (red if p < 0.05)', '<b>ADF</b> (green if p < 0.05)'])
for j, (says, p) in enumerate([('KPSS says', 'KPSS p'), ('ADF says', 'ADF p')], start=1):
    word = tests[says].unstack().loc[stores, depts]
    pval = tests[p].unstack().loc[stores, depts]
    text = word + '<br>p ' + pval.map('{:.3f}'.format)
    fig.add_trace(go.Heatmap(z=(word == 'Not stationary').astype(float).to_numpy(), x=depts, y=stores,
                             text=text.to_numpy(), texttemplate='%{text}', textfont=dict(size=11),
                             zmin=0, zmax=1, showscale=False, colorscale=[[0, '#8fd19e'], [1, '#e8a3a3']],
                             xgap=3, ygap=3, hovertemplate='%{y}, %{x}<br>%{text}<extra></extra>'), row=1, col=j)
fig.update_yaxes(autorange='reversed')
k_red, a_red = (tests['KPSS says'] == 'Not stationary').sum(), (tests['ADF says'] == 'Not stationary').sum()
fig.update_layout(title='<b>Are the 30 raw series stationary? Each test on its own (training data)</b><br>'
                        f'<span style="font-size:13px;color:#444">KPSS: {30 - k_red} green, {k_red} red. '
                        f'ADF: {30 - a_red} green, {a_red} red.</span>',
                  template='simple_white', height=600, margin=dict(t=110, l=150))
fig.show()


**What I see (draft)**
- KPSS: **all 30 red.** Every p is 0.02 or less. The level moves in every series.
- ADF: **12 green, 18 red.** The greens are the flatter ones (Texas store 1, California stores 1 and 3 dept 1/2).
- They don't fully agree, but KPSS (the book's test) says no everywhere, and ADF says no for most.
- Decision to make: weekly differencing or not.

## Step 3c: seasonal difference (lag 7) to make them stationary

In [ ]:
# Seasonal difference, lag 7: d(t) = sales today - sales on the same weekday last week. Then KPSS and ADF again.
train_d7 = (train - train.shift(7)).dropna()

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    kpss_p7 = pd.Series({c: kpss(train_d7[c], regression='c', nlags='auto')[1] for c in train_d7})
    adf_p7 = pd.Series({c: adfuller(train_d7[c], autolag='AIC')[1] for c in train_d7})

tests_d7 = pd.DataFrame({'KPSS p': kpss_p7, 'KPSS says': np.where(kpss_p7 < 0.05, 'Not stationary', 'Stationary'),
                         'ADF p': adf_p7, 'ADF says': np.where(adf_p7 < 0.05, 'Stationary', 'Not stationary')})
print(tests_d7[['KPSS says', 'ADF says']].apply(pd.Series.value_counts).fillna(0).astype(int))

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=['<b>KPSS</b> (red if p < 0.05)', '<b>ADF</b> (green if p < 0.05)'])
for j, (says, p) in enumerate([('KPSS says', 'KPSS p'), ('ADF says', 'ADF p')], start=1):
    word = tests_d7[says].unstack().loc[stores, depts]
    pval = tests_d7[p].unstack().loc[stores, depts]
    text = word + '<br>p ' + pval.map('{:.3f}'.format)
    fig.add_trace(go.Heatmap(z=(word == 'Not stationary').astype(float).to_numpy(), x=depts, y=stores,
                             text=text.to_numpy(), texttemplate='%{text}', textfont=dict(size=11),
                             zmin=0, zmax=1, showscale=False, colorscale=[[0, '#8fd19e'], [1, '#e8a3a3']],
                             xgap=3, ygap=3, hovertemplate='%{y}, %{x}<br>%{text}<extra></extra>'), row=1, col=j)
fig.update_yaxes(autorange='reversed')
k_red, a_red = (tests_d7['KPSS says'] == 'Not stationary').sum(), (tests_d7['ADF says'] == 'Not stationary').sum()
fig.update_layout(title='<b>After a seasonal difference (lag 7): are the 30 series stationary now?</b><br>'
                        f'<span style="font-size:13px;color:#444">KPSS: {30 - k_red} green, {k_red} red. '
                        f'ADF: {30 - a_red} green, {a_red} red. (KPSS p 0.10 means "0.10 or more".)</span>',
                  template='simple_white', height=600, margin=dict(t=110, l=150))
fig.show()

In [ ]:
# All 30 series AFTER the seasonal difference (lag 7): today minus the same weekday last week.
# Each point = average of that daily change over one week (daily is too spiky to read at this size).
# Stationary should look like: bouncing around the zero line, no drift up or down.
diff7 = (series - series.shift(7)).dropna()
weekly_d = diff7.resample('W').mean()

fig = make_subplots(rows=len(stores), cols=len(depts), shared_xaxes=True, shared_yaxes='columns',
                    column_titles=[f'<b>{d}</b>' for d in depts], row_titles=stores,
                    vertical_spacing=0.012, horizontal_spacing=0.04)
for r, store in enumerate(stores, start=1):
    for c, dept in enumerate(depts, start=1):
        y = weekly_d[(store, dept)]
        fig.add_trace(go.Scatter(x=y.index, y=y, mode='lines', showlegend=False,
                                 line=dict(color=STATE_COLOR[store.split(' store ')[0]], width=1.2),
                                 hovertemplate=f'{store}, {dept}<br>week of %{{x|%d %b %Y}}: '
                                               '%{y:+,.0f} vs same weekday last week<extra></extra>'), row=r, col=c)
        fig.add_hline(y=0, line=dict(color='#888', width=1), row=r, col=c)
fig.update_yaxes(nticks=3, tickfont=dict(size=9))
fig.update_annotations(selector=dict(textangle=90), textangle=0, xanchor='left', font=dict(size=10))
fig.update_layout(title='<b>The 30 series after a seasonal difference (lag 7)</b><br>'
                        '<span style="font-size:13px;color:#444">Change vs the same weekday last week, averaged per week. '
                        'Grey line = no change. Colour = state. Same scale down each column.</span>',
                  template='simple_white', height=1500, margin=dict(t=110, r=150))
fig.show()


**What I see (draft)**
- One seasonal difference and it's done. **KPSS: 30 of 30 green. ADF: 30 of 30 green.**
- Every KPSS p is 0.10 or more, every ADF p is basically 0. Both tests agree now, no unclear ones.
- So D = 1 at lag 7 is enough. No need for an extra first difference.
- Careful though (Step 9c): stationary doesn't mean better forecasts. Forecasting the change badly made it worse.

## Step 4: naive (the forecast is yesterday's value)

In [ ]:
# Naive: every day of the 28-day test gets the last value we saw (24 Apr 2016).
# We can't use "yesterday" inside the test window, because on the forecast day we don't know it yet.
naive = pd.DataFrame([train.iloc[-1]] * HORIZON, index=test.index)

# Plot: last 8 weeks of training (grey), actual test days (black), naive forecast (orange)
history = train.iloc[-56:]
fig = make_subplots(rows=10, cols=3, shared_xaxes=True, column_titles=depts, row_titles=stores,
                    vertical_spacing=0.012, horizontal_spacing=0.04)
for r, store in enumerate(stores, start=1):
    for c, dept in enumerate(depts, start=1):
        col = (store, dept)
        first = (r == 1 and c == 1)                              # show the legend once
        fig.add_trace(go.Scatter(x=history.index, y=history[col], name='before test', showlegend=first,
                                 line=dict(color='#bbb', width=1)), row=r, col=c)
        fig.add_trace(go.Scatter(x=test.index, y=test[col], name='actual', showlegend=first,
                                 line=dict(color='black', width=1.3)), row=r, col=c)
        fig.add_trace(go.Scatter(x=naive.index, y=naive[col], name='naive forecast', showlegend=first,
                                 line=dict(color=plots.PALETTE[1], width=2, dash='dash')), row=r, col=c)
fig.update_yaxes(rangemode='tozero', nticks=3, tickfont=dict(size=9))
fig.update_annotations(selector=dict(textangle=90), textangle=0, xanchor='left', font=dict(size=10))
fig.update_layout(title='<b>Naive forecast vs actual, 30 series</b><br>'
                        '<span style="font-size:13px;color:#444">Naive = the last known day (Sun 24 Apr 2016) repeated '
                        'for all 28 test days. Each panel has its own scale.</span>',
                  template='simple_white', height=1500, hovermode='x unified', margin=dict(t=110, r=150),
                  legend=dict(orientation='h', y=1.03, x=1, xanchor='right'))
fig.show()

**What I see (draft)**
- Naive is a flat line. The actual goes up and down every week. Flat can't follow that.
- Last known day is a **Sunday**, a busy day. So naive sits above most weekdays, below Saturdays.
- On average it lands close (median about 3% above the test average). But per series it's 39% under to 43% over.
- So the level is roughly right, the shape is all wrong. Seasonal naive should fix the shape.

## Step 5: pick the metrics
*(after reading Hyndman 5.8)*

- 30 series, very different sizes. ~2,000 a day vs ~180. So MAE alone can't compare them.
- MAPE is unit-free and fine here. No zero days at store × dept. Breaks at item level though.
- MASE: scale-free. Below 1 = better than "same weekday last week" was in training. Seasonal, so m = 7.
- RMSSE: same idea, big misses hurt more. M5 scores with a weighted version of it.
- MAE stays, just to read one series in units. Bias to see if we're too high or too low.

**Main metric:** MASE (m = 7), averaged over 30 series.
**Yardstick:** last year of training only. The 5-year one made the stores that grew look much worse (see side check).
**Also:** RMSSE, MAPE for the standup slide, MAE per series, bias.

## Step 6: score naive

In [ ]:
# One function scores any forecast against the test days, per series.
# Scale for MASE / RMSSE = how wrong "same weekday last week" was, on average, in the LAST YEAR of training (m = 7).
# Decided after the side check below: a 5-year scale made the stores that grew look far worse than they are.
M, SCALE_DAYS = 7, 365
recent = train.iloc[-SCALE_DAYS:]
scale_abs = (recent - recent.shift(M)).abs().mean()         # in-sample seasonal naive MAE, per series
scale_sq = ((recent - recent.shift(M)) ** 2).mean()         # same, squared (for RMSSE)

def score(forecast):
    err = test - forecast                                    # + = we forecast too low
    return pd.DataFrame({
        'MAE (units)': err.abs().mean(),
        'MAPE %': (err.abs() / test).mean() * 100,
        'MASE': err.abs().mean() / scale_abs,
        'RMSSE': np.sqrt((err ** 2).mean() / scale_sq),
        'bias %': -err.mean() / test.mean() * 100,           # + = over-forecast on average
    })

# Coloured score tables. Green = good, amber = so-so, red = worse than expected. MAE stays white (units, no fixed bar).
#   MASE, RMSSE: green below 1 (beats "same weekday last week"), amber up to 1.5, red above
#   MAPE %:      green below 15%, amber up to 25%, red above
#   bias %:      green within ±5%, amber within ±15%, red beyond
RULES = {'MASE': (1.0, 1.5), 'RMSSE': (1.0, 1.5), 'MAPE': (15, 25), 'bias': (5, 15)}
GREEN, AMBER, RED, WHITE = '#c8ecd0', '#fbe8bd', '#f4c6c6', 'white'

def _colour(metric, v):
    for key, (g, a) in RULES.items():
        if key in str(metric):
            v = abs(v)
            return GREEN if v < g else AMBER if v <= a else RED
    return WHITE

def _label(x):
    return ' · '.join(map(str, x)) if isinstance(x, tuple) else str(x)

def score_table(df, title=''):
    """Show a score table with coloured cells. Works with metrics as columns or as rows."""
    metrics_in_rows = any(k in str(i) for i in df.index for k in RULES)
    cols = [_label(c) for c in df.columns]
    values, colours = [[_label(i) for i in df.index]], [[WHITE] * len(df)]
    for c in df.columns:
        col = df[c]
        values.append([f'{v:,.2f}' for v in col])
        colours.append([_colour(i if metrics_in_rows else (c[-1] if isinstance(c, tuple) else c), v)
                        for i, v in col.items()])
    fig = go.Figure(go.Table(
        header=dict(values=[f'<b>{h}</b>' for h in [''] + cols], fill_color='#eeeeee', align='left', font=dict(size=12)),
        cells=dict(values=values, fill_color=colours, align='left', font=dict(size=12), height=26)))
    fig.update_layout(title=dict(text=f'<b>{title}</b><br><span style="font-size:12px;color:#555">green = good · '
                                      'amber = so-so · red = worse than expected · MAE in units, not coloured</span>'),
                      height=130 + 27 * len(df), margin=dict(t=80, l=10, r=10, b=10))
    fig.show()

naive_scores = score(naive)
score_table(naive_scores.mean().to_frame('naive, average of 30'), 'Naive: average over the 30 series')

In [ ]:
score_table(naive_scores.sort_values('MASE', ascending=False), 'Naive: every series, worst MASE on top')

**What I see (draft)**
- Naive MASE 1.60 on average. Worse than "same weekday last week" in the last training year.
- MAPE ~24%. Off by about a quarter on a typical day.
- Median MASE 1.29, so a few series drag the average up.
- Worst: California store 2, Food dept 2 (MASE 3.6, MAPE 53%). Copied a busy Sunday onto a series that just jumped.
- Only 7 of 30 below 1. Best: California store 4, Food dept 1 (0.84).

## Step 7: seasonal naive (7 days)

In [ ]:
# Seasonal naive (m = 7): each test day gets the value from the same weekday in the last known week.
# Last known week = 18-24 Apr 2016. Repeated 4 times to cover the 28 test days.
last_week = train.iloc[-7:]
snaive7 = pd.DataFrame(np.tile(last_week.to_numpy(), (HORIZON // 7, 1)), index=test.index, columns=train.columns)
print('test starts on', test.index[0].day_name(), '| last known week starts on', last_week.index[0].day_name())
snaive7.head(8).iloc[:, :3]

## Step 8: both forecasts against the actual test days

In [ ]:
# Both forecasts against the actual test days. Same layout as before.
fig = make_subplots(rows=10, cols=3, shared_xaxes=True, column_titles=depts, row_titles=stores,
                    vertical_spacing=0.012, horizontal_spacing=0.04)
for r, store in enumerate(stores, start=1):
    for c, dept in enumerate(depts, start=1):
        col = (store, dept)
        first = (r == 1 and c == 1)
        fig.add_trace(go.Scatter(x=history.index, y=history[col], name='before test', showlegend=first,
                                 line=dict(color='#bbb', width=1)), row=r, col=c)
        fig.add_trace(go.Scatter(x=test.index, y=test[col], name='actual', showlegend=first,
                                 line=dict(color='black', width=1.3)), row=r, col=c)
        fig.add_trace(go.Scatter(x=naive.index, y=naive[col], name='naive', showlegend=first,
                                 line=dict(color=plots.PALETTE[1], width=1.5, dash='dash')), row=r, col=c)
        fig.add_trace(go.Scatter(x=snaive7.index, y=snaive7[col], name='seasonal naive (7 days)', showlegend=first,
                                 line=dict(color=plots.PALETTE[0], width=1.5, dash='dot')), row=r, col=c)
fig.update_yaxes(rangemode='tozero', nticks=3, tickfont=dict(size=9))
fig.update_annotations(selector=dict(textangle=90), textangle=0, xanchor='left', font=dict(size=10))
fig.update_layout(title='<b>Naive vs seasonal naive vs actual, 30 series</b><br>'
                        '<span style="font-size:13px;color:#444">Naive = last day repeated. Seasonal naive = last week '
                        'repeated (same weekday). Each panel has its own scale.</span>',
                  template='simple_white', height=1500, hovermode='x unified', margin=dict(t=110, r=150),
                  legend=dict(orientation='h', y=1.03, x=1, xanchor='right'))
fig.show()

**What I see (draft)**
- Seasonal naive has the weekly up-and-down. Naive is flat. You can see it straight away.
- But seasonal naive sits **below** the actual in most panels, mostly in the first half of May.

## Step 9: score both and compare

In [ ]:
# Score both with the same function, then compare.
snaive7_scores = score(snaive7)
compare = pd.DataFrame({'naive (mean)': naive_scores.mean(), 'seasonal naive 7 (mean)': snaive7_scores.mean(),
                        'naive (median)': naive_scores.median(), 'seasonal naive 7 (median)': snaive7_scores.median()})
wins = (snaive7_scores['MASE'] < naive_scores['MASE']).sum()
print(f'Seasonal naive beats naive on MASE in {wins} of 30 series')
score_table(compare, 'Naive vs seasonal naive 7')

In [ ]:
# Per series, side by side (sorted by seasonal naive MASE, worst on top)
side = pd.concat({'naive': naive_scores[['MASE', 'MAPE %']], 'seasonal naive 7': snaive7_scores[['MASE', 'MAPE %']]},
                 axis=1).sort_values(('seasonal naive 7', 'MASE'), ascending=False)
score_table(side, 'Per series: naive vs seasonal naive 7')

**What I see (draft)**
- Seasonal naive wins clearly. MASE 1.60 → 1.06, MAPE 24% → 16%. Better in 21 of 30 series.
- Median MASE ~0.98, about the same as "last week" was in training. Makes sense, it's the same rule.
- But it's **too low** in 28 of 30 series (about −12%). The week it copies (18-24 Apr) was a quiet one.
- Errors peak in test weeks 2 and 3 (2-15 May) = SNAP days. Last week of April had no SNAP.
- So the weekly shape is right, the monthly SNAP lift is missing. That's the case for trying 30 days next.

### Side check: MASE / RMSSE with a 1-year yardstick (seasonal naive only)

In [ ]:
# Same seasonal naive forecast, scored with two yardsticks: all 5+ years of training vs the last year only.
# Only the yardstick changes; the forecast and its errors stay the same.
diff_5y = train - train.shift(M)
err = test - snaive7
scale_check = pd.DataFrame({
    'MASE (5 yr scale)': err.abs().mean() / diff_5y.abs().mean(),
    'MASE (1 yr scale)': err.abs().mean() / scale_abs,
    'RMSSE (5 yr scale)': np.sqrt((err ** 2).mean() / (diff_5y ** 2).mean()),
    'RMSSE (1 yr scale)': np.sqrt((err ** 2).mean() / scale_sq),
})
print('Average over 30 series:'); print(scale_check.mean().round(2).to_string())
score_table(scale_check.sort_values('MASE (5 yr scale)', ascending=False), 'Seasonal naive 7: 5-year vs 1-year yardstick')

In [ ]:
# One chart: each row is one series. Grey dot = 5-year scale, blue dot = 1-year scale. Line = how much it moved.
order = scale_check.sort_values('MASE (1 yr scale)').index
labels = [f'{s} · {d}' for s, d in order]
five, one = scale_check.loc[order, 'MASE (5 yr scale)'], scale_check.loc[order, 'MASE (1 yr scale)']
fig = go.Figure()
for lab, a, b in zip(labels, five, one):
    fig.add_trace(go.Scatter(x=[a, b], y=[lab, lab], mode='lines', line=dict(color='#ddd', width=3),
                             showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=five, y=labels, mode='markers', name='scale from all 5 years of training',
                         marker=dict(color='#999', size=9, symbol='circle-open', line=dict(width=2)),
                         hovertemplate='%{y}<br>5-year scale: MASE %{x:.2f}<extra></extra>'))
fig.add_trace(go.Scatter(x=one, y=labels, mode='markers', name='scale from the last year only',
                         marker=dict(color=plots.PALETTE[0], size=9),
                         hovertemplate='%{y}<br>1-year scale: MASE %{x:.2f}<extra></extra>'))
fig.add_vline(x=1, line=dict(color='#333', dash='dot'),
              annotation_text='1 = as good as "same weekday last week" in training', annotation_position='top right')
fig.update_layout(title='<b>Seasonal naive: MASE with a 5-year vs a 1-year yardstick</b><br>'
                        '<span style="font-size:13px;color:#444">Same forecast, same errors. Only the scale changes. '
                        'Long lines = series that grew a lot, so the old scale made them look worse.</span>',
                  xaxis_title='MASE (lower is better)', template='simple_white', height=820,
                  margin=dict(t=110, l=260), legend=dict(orientation='h', y=-0.08, x=0.5, xanchor='center'))
fig.show()

**What I see (draft)**
- Same forecast, only the yardstick changes. Average MASE 1.19 → 1.06 with the last-year scale.
- Biggest drop: California store 2, Food dept 2 (2.40 → 1.12). Then Wisconsin stores 1 and 2, Food dept 2. The series that jumped.
- Flat series barely move. Some even get a bit worse (California store 3), their last year was calmer.
- So the 5-year yardstick punishes the stores that grew. The forecast isn't as bad as 2.40 said.
- Still need to pick one scale and keep it for every model.

## Step 9b: seasonal naive (30 days)
Same date last month. Idea: catch the SNAP lift that 7 days missed.

In [ ]:
# Seasonal naive (30 days): each test day gets the value from 30 days earlier.
# 30 > 28, so every value it needs is still inside the training data.
snaive30 = series.shift(30).loc[test.index]
pd.DataFrame({'test day': test.index.strftime('%a %d %b'),
              'copied from': (test.index - pd.Timedelta(days=30)).strftime('%a %d %b')}).head(7)

In [ ]:
# Actual vs seasonal naive 7 vs seasonal naive 30. Same layout as before.
fig = make_subplots(rows=10, cols=3, shared_xaxes=True, column_titles=depts, row_titles=stores,
                    vertical_spacing=0.012, horizontal_spacing=0.04)
for r, store in enumerate(stores, start=1):
    for c, dept in enumerate(depts, start=1):
        col = (store, dept)
        first = (r == 1 and c == 1)
        fig.add_trace(go.Scatter(x=history.index, y=history[col], name='before test', showlegend=first,
                                 line=dict(color='#bbb', width=1)), row=r, col=c)
        fig.add_trace(go.Scatter(x=test.index, y=test[col], name='actual', showlegend=first,
                                 line=dict(color='black', width=1.3)), row=r, col=c)
        fig.add_trace(go.Scatter(x=snaive7.index, y=snaive7[col], name='seasonal naive (7 days)', showlegend=first,
                                 line=dict(color=plots.PALETTE[0], width=1.5, dash='dot')), row=r, col=c)
        fig.add_trace(go.Scatter(x=snaive30.index, y=snaive30[col], name='seasonal naive (30 days)', showlegend=first,
                                 line=dict(color=plots.PALETTE[3], width=1.5, dash='dash')), row=r, col=c)
fig.update_yaxes(rangemode='tozero', nticks=3, tickfont=dict(size=9))
fig.update_annotations(selector=dict(textangle=90), textangle=0, xanchor='left', font=dict(size=10))
fig.update_layout(title='<b>Seasonal naive 7 days vs 30 days vs actual, 30 series</b><br>'
                        '<span style="font-size:13px;color:#444">7 days = same weekday last week. '
                        '30 days = same date last month (weekday shifts by 2). Each panel has its own scale.</span>',
                  template='simple_white', height=1500, hovermode='x unified', margin=dict(t=110, r=150),
                  legend=dict(orientation='h', y=1.03, x=1, xanchor='right'))
fig.show()

In [ ]:
# Score all three with the same function (1-year yardstick)
snaive30_scores = score(snaive30)
all_scores = pd.DataFrame({'naive': naive_scores.mean(), 'seasonal naive 7': snaive7_scores.mean(),
                           'seasonal naive 30': snaive30_scores.mean()})
print('Mean over 30 series. MASE / RMSSE: below 1 = better than "same weekday last week" in the last training year')
print('seasonal naive 30 beats seasonal naive 7 (MASE) in',
      (snaive30_scores['MASE'] < snaive7_scores['MASE']).sum(), 'of 30 series')
score_table(all_scores, 'Naive vs seasonal naive 7 vs seasonal naive 30 (average of 30)')

**What I see (draft)**
- 30 days is worse. MASE 1.71, even worse than plain naive (1.60). Beats 7 days in only 6 of 30.
- Why: 30 days = 4 weeks + 2 days. Every Monday gets copied from a Saturday. The weekly shape lands 2 days off.
- Worst in test week 1 (MAPE 29%), where a busy weekend gets copied onto quiet weekdays.
- So the weekly rhythm matters more than SNAP. Lose the weekday and you lose more than SNAP gives back.
- Still too low on average (−8%). The month it copies (late Mar to late Apr) was quieter than May.

## Step 9c: forecast the weekly differences instead
- KPSS says all 30 raw series move. Weekly differencing makes them stable.
- So: turn each series into "change vs same weekday last week", forecast that, add it back, score in units.
- Naive on the change = last week's Sunday-to-Sunday change, repeated.
- Seasonal naive on the change = last week's change for each weekday, repeated.
- Forecast starts from the last real week (18-24 Apr), then each forecast week builds on the one before.

In [ ]:
# Weekly differencing: d(t) = sales today - sales on the same weekday last week.
# We forecast d, then undo the differencing to get units back, then score in units like every other model.
dtrain = (train - train.shift(7)).dropna()

# Naive on d: every future d = the LAST d we saw (Sun 24 Apr minus Sun 17 Apr)
naive_d = pd.DataFrame([dtrain.iloc[-1]] * HORIZON, index=test.index)
# Seasonal naive on d: every future d = the d of the same weekday in the last known week
snaive_d = pd.DataFrame(np.tile(dtrain.iloc[-7:].to_numpy(), (HORIZON // 7, 1)), index=test.index, columns=train.columns)

def undo_weekly_diff(d_forecast):
    """Units = same weekday one week earlier + forecast change. Week 1 builds on the last real week,
    week 2 builds on forecast week 1, and so on."""
    prev = train.iloc[-7:].to_numpy()
    weeks = []
    for k in range(HORIZON // 7):
        prev = prev + d_forecast.iloc[7 * k: 7 * (k + 1)].to_numpy()
        weeks.append(prev)
    return pd.DataFrame(np.vstack(weeks), index=test.index, columns=train.columns)

# Seasonal naive 30 on d: every future d = the d from 30 days earlier (still inside training)
snaive30_d = (series - series.shift(7)).shift(30).loc[test.index]

naive_on_d = undo_weekly_diff(naive_d)
snaive_on_d = undo_weekly_diff(snaive_d)
snaive30_on_d = undo_weekly_diff(snaive30_d)
print('Last known change (Sun 24 Apr vs Sun 17 Apr), first 3 series:'); dtrain.iloc[-1].head(3).round(0)

In [ ]:
# Compare, in units, with the same scoring function and the same 1-year yardstick
diff_scores = {'naive (raw)': naive_scores, 'seasonal naive 7 (raw)': snaive7_scores,
               'naive on weekly diff': score(naive_on_d), 'seasonal naive on weekly diff': score(snaive_on_d),
               'seasonal naive 30 on weekly diff': score(snaive30_on_d)}
diff_compare = pd.DataFrame({k: v.mean() for k, v in diff_scores.items()})
print('negative forecasts (units < 0):', int((naive_on_d < 0).sum().sum()), 'naive on diff,',
      int((snaive_on_d < 0).sum().sum()), 'seasonal naive on diff')
score_table(diff_compare, 'Raw vs weekly-differenced (average of 30)')

In [ ]:
# One chart: average MASE per model, raw vs weekly-differenced
names = list(diff_scores)
vals = [diff_scores[n]['MASE'].mean() for n in names]
fig = go.Figure(go.Bar(x=[n.replace(' (raw)', '<br>(raw)').replace(' on weekly diff', '<br>on weekly diff') for n in names],
                       y=vals, marker_color=['#9aa5ad', plots.PALETTE[0], '#c9a3b8', plots.PALETTE[3], '#d9c3cf'],
                       text=[f'{v:.2f}' for v in vals], textposition='outside',
                       hovertemplate='%{x}<br>MASE %{y:.2f}<extra></extra>'))
fig.add_hline(y=1, line=dict(color='#333', dash='dot'))
fig.update_layout(title='<b>Does forecasting the weekly differences help?</b><br>'
                        '<span style="font-size:13px;color:#444">Average MASE over 30 series, scored in units. Lower is better.</span>',
                  yaxis=dict(title='average MASE', range=[0, max(vals) * 1.2]), template='simple_white', height=440,
                  margin=dict(t=100))
fig.show()

In [ ]:
# Differenced forecasts back in units vs actual. Same layout as before.
fig = make_subplots(rows=10, cols=3, shared_xaxes=True, column_titles=depts, row_titles=stores,
                    vertical_spacing=0.012, horizontal_spacing=0.04)
for r, store in enumerate(stores, start=1):
    for c, dept in enumerate(depts, start=1):
        col = (store, dept)
        first = (r == 1 and c == 1)
        fig.add_trace(go.Scatter(x=history.index, y=history[col], name='before test', showlegend=first,
                                 line=dict(color='#bbb', width=1)), row=r, col=c)
        fig.add_trace(go.Scatter(x=test.index, y=test[col], name='actual', showlegend=first,
                                 line=dict(color='black', width=1.3)), row=r, col=c)
        fig.add_trace(go.Scatter(x=snaive7.index, y=snaive7[col], name='seasonal naive 7 (raw)', showlegend=first,
                                 line=dict(color=plots.PALETTE[0], width=1.5, dash='dot')), row=r, col=c)
        fig.add_trace(go.Scatter(x=naive_on_d.index, y=naive_on_d[col], name='naive on weekly diff', showlegend=first,
                                 line=dict(color=plots.PALETTE[1], width=1.5, dash='dash')), row=r, col=c)
        fig.add_trace(go.Scatter(x=snaive_on_d.index, y=snaive_on_d[col], name='seasonal naive on weekly diff',
                                 showlegend=first, line=dict(color=plots.PALETTE[3], width=1.5, dash='dashdot')),
                      row=r, col=c)
fig.update_yaxes(nticks=3, tickfont=dict(size=9))
fig.update_annotations(selector=dict(textangle=90), textangle=0, xanchor='left', font=dict(size=10))
fig.update_layout(title='<b>Forecasting the weekly change, then adding it back: 30 series</b><br>'
                        '<span style="font-size:13px;color:#444">Watch the dashed lines drift further away each week: '
                        'last week\'s change is added again and again. Each panel has its own scale.</span>',
                  template='simple_white', height=1500, hovermode='x unified', margin=dict(t=110, r=150),
                  legend=dict(orientation='h', y=1.03, x=1, xanchor='right'))
fig.show()

In [ ]:
# Per series: MASE and MAPE for the two differenced models next to plain seasonal naive 7
side_d = pd.concat({'seasonal naive 7 (raw)': snaive7_scores[['MASE', 'MAPE %']],
                    'naive on weekly diff': diff_scores['naive on weekly diff'][['MASE', 'MAPE %']],
                    'seasonal naive on weekly diff': diff_scores['seasonal naive on weekly diff'][['MASE', 'MAPE %']]},
                   axis=1).sort_values(('seasonal naive on weekly diff', 'MASE'), ascending=False)
score_table(side_d, 'Per series: raw seasonal naive 7 vs the two differenced models')

**What I see (draft)**
- Differencing made naive and seasonal naive 7 **worse**, a lot. MASE 2.43 and 2.52 vs 1.06 for plain seasonal naive.
- Only seasonal naive 30 got better on the differences (1.71 → 1.55). Adding back onto the last real week fixes its 2-day weekday shift. Still far behind 1.06.
- Why: both copy last week's *change* and keep adding it, 4 weeks in a row. One odd week gets multiplied by 4.
- Some forecasts even go below zero (53 days in seasonal naive on diff).
- Funny thing: if you forecast the change as **0**, you get plain seasonal naive 7 back. Which is the best one.
- So differencing isn't the win. What you forecast for the change is. Next try: the *average* change (drift).

## Step 10: what does the gap say about the weekly rhythm?
1. Naive → seasonal naive 7: MASE 1.60 → 1.06, MAPE 24% → 16%. Just adding the weekday removes about a third of the error.
2. Seasonal naive 30 breaks the weekday (2 days off) and is the worst of all (1.71). So the weekly rhythm is worth more than the monthly SNAP lift.
3. What's left is level, not shape: seasonal naive 7 is ~12% too low, worst in the SNAP weeks. Next model should keep the weekday and add a level / SNAP signal.

## Step 11: Bottomline
- **Level:** store × food dept, 30 daily series. Middle-out: add up to store and total later.
- **Test:** last 28 days (25 Apr to 22 May 2016), trained on everything before.
- **Metric:** MASE (m = 7), yardstick from the last training year. MAPE alongside for a plain %.
- **Scores (avg of 30):** naive 1.60 (~24% off a day) · seasonal naive 7 **1.06** (~16%) · seasonal naive 30 1.71 (~24%).
- **Takeaway:** keep the weekday. Seasonal naive 7 is the bar to beat. It runs ~12% low, mostly in SNAP weeks.
- **Next:** a model that keeps the weekly shape but tracks the level (e.g. exponential smoothing), plus SNAP.
- **Chart:** below.

In [ ]:
# Standup chart: average MASE over the 30 series. Pairs = same model on raw data vs on weekly-differenced data.
groups = ['Naive<br>(last day)', 'Seasonal naive 7<br>(same weekday last week)', 'Seasonal naive 30<br>(same date last month)']
raw = [naive_scores['MASE'].mean(), snaive7_scores['MASE'].mean(), snaive30_scores['MASE'].mean()]
dif = [diff_scores['naive on weekly diff']['MASE'].mean(), diff_scores['seasonal naive on weekly diff']['MASE'].mean(),
       diff_scores['seasonal naive 30 on weekly diff']['MASE'].mean()]
fig = go.Figure([
    go.Bar(name='on raw data', x=groups, y=raw, marker_color=plots.PALETTE[0],
           text=[f'{v:.2f}' for v in raw], textposition='outside', hovertemplate='%{x}, raw<br>MASE %{y:.2f}<extra></extra>'),
    go.Bar(name='on weekly-differenced data', x=groups, y=dif, marker_color=plots.PALETTE[1],
           marker_pattern_shape='/', text=[f'{v:.2f}' for v in dif], textposition='outside',
           hovertemplate='%{x}, weekly-differenced<br>MASE %{y:.2f}<extra></extra>')])
fig.add_hline(y=1, line=dict(color='#333', dash='dot'),
              annotation_text='1 = as good as "same weekday last week" in the last training year', annotation_position='top left')
fig.update_layout(title='<b>Seasonal naive 7 on raw data is still the baseline to beat</b><br>'
                        '<span style="font-size:13px;color:#444">30 store × food department series, test 25 Apr to 22 May 2016. '
                        'Average MASE, lower is better.</span>',
                  barmode='group', bargap=0.3, yaxis=dict(title='average MASE (lower is better)', range=[0, max(raw + dif) * 1.2]),
                  template='simple_white', height=480, margin=dict(t=110), legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center'))
fig.show()

# Part 2: look at 5 items (learning only)
Not our forecast level. Just to see what happens to the noise when we add items up.

## Step 12: pick 5 items
**Rule:**
- Food dept 3 only (the biggest, so the sums are readable).
- Rank items by units sold in the last training year, all 10 stores together.
- Keep only items that sold at least once in every store that year (otherwise MASE divides by zero).
- Best = top 2. Typical = the middle one. Least = bottom 2.

In [ ]:
PLOT_START = '2016-02-01'      # same time frame rule as the clean notebook
ITEM_DEPT = 'Food dept 3'
last_year = days[-HORIZON - SCALE_DAYS:-HORIZON]

dept_rows = sales[sales['dept'] == ITEM_DEPT]
units = dept_rows.set_index(['item_id', 'store'])[last_year].sum(axis=1)       # last-year units, per item per store
sold_everywhere = (units.unstack() > 0).all(axis=1)
ranked = units.groupby('item_id').sum()[sold_everywhere].sort_values(ascending=False)
mid = len(ranked) // 2
picked = {'best 1': ranked.index[0], 'best 2': ranked.index[1], 'typical': ranked.index[mid],
          'least 2': ranked.index[-2], 'least 1': ranked.index[-1]}
item_name = {iid: f'{role}: item {iid[-3:]}' for role, iid in picked.items()}

rows = sales[sales['item_id'].isin(picked.values())].set_index(['item_id', 'store'])
item_store = rows[days].T.set_axis(idx)                                         # 50 series: 5 items x 10 stores
item_store.columns = pd.MultiIndex.from_tuples([(item_name[i], s) for i, s in item_store.columns], names=['item', 'store'])
item_store = item_store[[(item_name[i], s) for i in picked.values() for s in stores]]

ly = item_store.loc[train.index[-SCALE_DAYS:]]
print(f'{len(ranked)} of {dept_rows.item_id.nunique()} items in {ITEM_DEPT} sold in every store last year')
pd.DataFrame({'units a day, per store': ly.mean().groupby(level=0).mean(),
              'zero days %': (ly == 0).mean().groupby(level=0).mean() * 100}).loc[list(item_name.values())].round(1)

## Step 13: plot each item

In [ ]:
# One row per item. Thin lines = the 10 stores. Black = the average store. Grey band = the 28 test days.
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=[f'<b>{n}</b>' for n in item_name.values()])
for r, name in enumerate(item_name.values(), start=1):
    one = item_store[name].loc[PLOT_START:]
    for store in stores:
        fig.add_trace(go.Scatter(x=one.index, y=one[store], mode='lines', showlegend=False, opacity=0.45,
                                 line=dict(color=STATE_COLOR[store.split(' store ')[0]], width=0.8),
                                 hovertemplate=f'{store}<br>%{{x|%a %d %b}}: %{{y}}<extra></extra>'), row=r, col=1)
    fig.add_trace(go.Scatter(x=one.index, y=one.mean(axis=1), mode='lines', name='average store', showlegend=r == 1,
                             line=dict(color='black', width=2),
                             hovertemplate='average store<br>%{x|%a %d %b}: %{y:.1f}<extra></extra>'), row=r, col=1)
fig.add_vrect(x0=test.index[0], x1=test.index[-1], fillcolor='#e6e6e6', opacity=0.6, layer='below', line_width=0,
              row='all', col='all')
fig.update_yaxes(rangemode='tozero', nticks=4)
fig.update_layout(title=f'<b>5 items from {ITEM_DEPT}, units a day</b><br><span style="font-size:13px;color:#444">'
                        f'{pd.Timestamp(PLOT_START):%d %b} to {test.index[-1]:%d %b %Y}. Thin lines = each store '
                        '(colour = state). Each row has its own scale.</span>',
                  template='simple_white', height=1100, margin=dict(t=110),
                  legend=dict(orientation='h', y=1.04, x=1, xanchor='right'))
fig.show()

## Step 14: naive and seasonal naive on each item, MASE

In [ ]:
def baselines(tr):
    """Naive and seasonal naive 7 for every column of a training frame."""
    return {'naive': pd.DataFrame([tr.iloc[-1]] * HORIZON, index=test.index),
            'seasonal naive 7': pd.DataFrame(np.tile(tr.iloc[-M:].to_numpy(), (HORIZON // M, 1)),
                                             index=test.index, columns=tr.columns)}

def mase_of(actual, forecast):
    """MASE per column. Scale = last training year, m = 7 (same as Part 1)."""
    tr, te = actual.iloc[:-HORIZON], actual.iloc[-HORIZON:]
    recent = tr.iloc[-SCALE_DAYS:]
    return (te - forecast).abs().mean() / (recent - recent.shift(M)).abs().mean()

item_fc = baselines(item_store.iloc[:-HORIZON])
item_mase = pd.DataFrame({m: mase_of(item_store, f) for m, f in item_fc.items()})
print('MASE per item = average over its 10 stores')
item_mase.groupby(level=0).mean().loc[list(item_name.values())].round(2)

## Step 15: add the item forecasts up
- **Store:** the 5 items added up inside each store (10 series).
- **Dept:** the 5 items added up over all 10 stores (1 series). Only these 5 items, not the full Food dept 3.
- Forecasts are added up the same way. Then MASE again, with each level's own scale.

In [ ]:
LEVELS = {'item in one store\n(50 series)': lambda df: df,
          'store: 5 items added\n(10 series)': lambda df: df.T.groupby(level='store').sum().T,
          'dept: 5 items, all stores\n(1 series)': lambda df: df.sum(axis=1).to_frame('all')}

level_mase, level_noise = {}, {}
for level, add_up in LEVELS.items():
    actual = add_up(item_store)
    level_mase[level] = {m: mase_of(actual, add_up(f)).mean() for m, f in item_fc.items()}
    recent = actual.loc[train.index[-SCALE_DAYS:]]
    level_noise[level] = (recent.std() / recent.mean()).mean() * 100      # day-to-day spread, % of the average day
level_mase = pd.DataFrame(level_mase).T
print('Day-to-day spread (std / mean, last training year):',
      ' | '.join(f"{k.splitlines()[0]} {v:.0f}%" for k, v in level_noise.items()))
level_mase.round(2)

## Step 16: does the noise grow or shrink?

In [ ]:
lv = [k.replace('\n', '<br>') for k in level_mase.index]
fig = go.Figure([go.Bar(name=m, x=lv, y=level_mase[m], text=level_mase[m].map('{:.2f}'.format), textposition='outside',
                        marker_color=c, marker_pattern_shape=p, hovertemplate='%{x}<br>MASE %{y:.2f}<extra>' + m + '</extra>')
                 for m, c, p in [('naive', plots.PALETTE[1], ''), ('seasonal naive 7', plots.PALETTE[0], '/')]])
fig.add_hline(y=1, line=dict(color='#333', dash='dot'), annotation_text='1 = same as last week, last year',
              annotation_position='top left')
fig.update_layout(title='<b>Adding items up: does the error shrink?</b><br><span style="font-size:13px;color:#444">'
                        'Average MASE at each level. Left = single items, right = all 5 items in all stores. '
                        'Lower is better.</span>', barmode='group', bargap=0.3, template='simple_white', height=460,
                  yaxis=dict(title='average MASE', range=[0, level_mase.to_numpy().max() * 1.25]), margin=dict(t=110),
                  legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center'))
fig.show()

**What this tells me**
- One item in one store is mostly noise. Day-to-day spread is 4x its average day. The least sellers are 97% zeros, and MASE blows up (8 to 25).
- Add the 5 items up and the noise drops fast. Seasonal naive 7: 7.33 per item → 0.80 per store → 0.51 for all stores.
- So forecast where the signal is (store × dept). Items come later, split down, with a method built for zeros.